## MONAI Integration
MONAI allows the definition of AI models using the "bundle" concept. It allows for easy experimentation and sharing of models that have been developed using MONAI. Using the bundle configurations, we can use MONAI's MonaiAlgo (the implementation of ClientAlgo) to execute a bundle model in a federated scenario using NVFlare.

You can find the code for the [MONAI Integration code](https://github.com/NVIDIA/NVFlare/blob/dev/integration/monai/README.md) in the `integration` directory of the NVFlare GitHub.

There is also an example that walks through preparing the environment and input datasets, and running the [MONAI Spleen CT Segmentation example](https://github.com/NVIDIA/NVFlare/tree/dev/integration/monai/examples/spleen_ct_segmentation) using a locally provisioned secure deployment.  We'll walk through that here.

## Setting up the MONAI example

The MONAI example for a distributed (or local Docker Compose "distriburted" deployment) can be found in the `NVFlare/integration/monai/examples/spleen_ct_segmentation_real-world` directory.    Let's copy it to our `notebooks/examples` directory.

In [ ]:
!if [ ! -d examples ]; then mkdir examples; fi
!if [ ! -d examples/spleen_ct_segmentation_real-world ]; then \
    cp -r ../NVFlare/integration/monai/examples/spleen_ct_segmentation_real-world examples/; fi
!tree examples/spleen_ct_segmentation_real-world

We now need to download the MONAI bundle that contains the `spleen_ct_segmentation` model and configuration.  We can do this using the MONAI bundle download built-in script.

This model and configuration will be part of the FLARE spleen_ct_segmentation_real-world app that is deployed to the FLARE clients, so we'll provde the job directory as the bundle download path.

We will also download the example spleen dataset and push it to a directory accessible by the containers running the FLARE clients.

In [ ]:
%env JOB_DIR=examples/spleen_ct_segmentation_real-world/job
!python3 -m monai.bundle download \
    --name "spleen_ct_segmentation" \
    --version "0.3.7" \
    --bundle_dir ./${JOB_DIR}/app/config
!if [ ! -d data/Task09_Spleen ]; then \
    python3 examples/spleen_ct_segmentation_real-world/download_spleen_dataset.py; fi
!for site in site-1 site-2; do mkdir ${HOST_PATH}/${site}/data; cp -r data/Task09_Spleen ${HOST_PATH}/${site}/data; done

### Connecting to the Docker Compose deployment and running the MONAI app
Even though we're running locally via Docker compose, the server and all clients are running securely in their own container instance.  Compared to POC mode, where we were running without authentication enabled, we will need to use the `admin@nvidia.com` toolkit to establish a secure session, providing the admin username `admin@nvidia.com` and the certificates provided in the `monai_workspace/monai_demo/prod_00/admin@nvidia.com` directory.

In [ ]:
admin_dir = "monai_workspace/monai_demo/prod_00/admin@nvidia.com"
admin_user = "admin@nvidia.com"
from nvflare.fuel.flare_api.flare_api import new_secure_session

admin_session = new_secure_session(
    username = admin_user,
    startup_kit_location = admin_dir
)
print(admin_session.get_system_info())

In [ ]:
path_to_job_config = "/flare/notebooks/examples/spleen_ct_segmentation_real-world/job"
job_id = admin_session.submit_job(path_to_job_config)
print(job_id)

In [ ]:
import json

# Job Status
jobs_output = admin_session.list_jobs()
jobs_detail = admin_session.list_jobs(detailed=True)
print("Job Status")
print(json.dumps((jobs_output), indent=2))
print("\nJob Detail")
print(json.dumps((jobs_detail), indent=2))

# Job Metadata
print("\nJob Metadata")
admin_session.get_job_meta(job_id)